# Level 2: many games, over the months

Level 1 measured one game. The same measures over every game, grouped by the month (or the week)
the games were played: how many games, the balance of the matchups, how long the games last, the
workers and the income at fixed minutes, when the upgrades finish and which maps were in the pool.
The result of every query is a time series per entity (a matchup, a race, a map), the format
Level 3 needs to find the trends.

Every question is answered twice, with [polars](https://docs.pola.rs/) expressions and with
[DuckDB](https://duckdb.org/docs/) SQL over the same files; `s2.same()` checks that both return the
same rows, so each pair is a worked translation between the two.

Previous: [Level 1](./Level_1_One_Game.ipynb). Next: [Level 3](./Level_3_Trends.ipynb).

In [1]:
import polars as pl
import plotly.express as px

import snapshot as s2
import viz

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(200)
pl.Config.set_fmt_str_lengths(60)

# The snapshot: `$S2_IPC_DIR`, or `ipcs` in the replay directory when written by swarmy.
IPC_DIR = s2.ipc_dir()
# polars reads the files lazily, one LazyFrame per file.
details = s2.scan(IPC_DIR, "details")
stats = s2.scan(IPC_DIR, "stats")
upgrades = s2.scan(IPC_DIR, "upgrades")
unit_born = s2.scan(IPC_DIR, "unit_born")
unit_died = s2.scan(IPC_DIR, "unit_died")
# DuckDB reads the same files as SQL views of the same names, plus `players` and `games_1v1`.
con = s2.duck(IPC_DIR)

# One row per player of every game, the 1v1 games with the matchup, and one row per 1v1 game with
# the races sorted (PvT, never TvP), the winner and the length. The SQL views have the same names.
# They read every unit_born file, so they are collected once and reused.
players = s2.players(IPC_DIR).collect().lazy()
games_1v1 = s2.one_vs_one(IPC_DIR).collect().lazy()
games = s2.games(IPC_DIR).collect().lazy()

## Games per month

`dt.truncate("1mo")` puts every game in its month, `date_trunc('month', ...)` in SQL. The count of
distinct ids counts the games, not the players.

In [2]:
per_month = (
    details.group_by(pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(
        pl.col("ext_fs_id").n_unique().alias("games"),
        pl.col("title").n_unique().alias("maps"),
        pl.col("player_name").n_unique().alias("player_names"),
    )
    .sort("month")
    .collect()
)
per_month

month,games,maps,player_names
datetime[ns],u32,u32,u32
2019-08-01 00:00:00,218,7,103
2020-07-01 00:00:00,130,39,214
2020-08-01 00:00:00,923,79,933
2020-09-01 00:00:00,1022,59,720
2020-10-01 00:00:00,1379,78,990
2020-11-01 00:00:00,954,42,890
2020-12-01 00:00:00,630,42,653
2021-01-01 00:00:00,813,54,861
2021-02-01 00:00:00,797,40,646


In [3]:
sql = con.sql("""
    SELECT
        date_trunc('month', ext_datetime) AS month,
        count(DISTINCT ext_fs_id) AS games,
        count(DISTINCT title) AS maps,
        count(DISTINCT player_name) AS player_names
    FROM details
    GROUP BY ALL
    ORDER BY month
""")
s2.same(per_month, sql)

'same 76 rows, 4 columns'

In [4]:
fig = px.bar(per_month, x="month", y="games", title="Games per month in the snapshot")
fig.update_traces(marker_line_width=0)
viz.show(fig, "level_2_games_per_month")

## The balance of the matchups

The win rate of the first race of each matchup per month, `PvT` is the share of the games the
Protoss won against Terran. Only the games of two different races (the mirrors are 50% by
definition) and the months with at least 10 games of the matchup: a win rate of 10 games moves by
15 points by chance, which is why Level 3 looks at the trend and not at a month.

`games_1v1` has two rows per game, one per player; the filter `race < opponent_race` keeps the
row of the first race of the matchup, one per game.

In [5]:
MIN_GAMES = 10
balance = (
    games_1v1.filter((pl.col("race") < pl.col("opponent_race")) & (pl.col("race") != "?") & (pl.col("opponent_race") != "?"))
    .group_by("matchup", pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("games"), (pl.col("player_result") == "Win").mean().alias("win_rate"))
    .filter(pl.col("games") >= MIN_GAMES)
    .sort("matchup", "month")
    .collect()
)
balance.head(8)

matchup,month,games,win_rate
str,datetime[ns],u32,f64
"""PvT""",2019-08-01 00:00:00,39,0.589744
"""PvT""",2020-07-01 00:00:00,14,0.5
"""PvT""",2020-08-01 00:00:00,117,0.521368
"""PvT""",2020-09-01 00:00:00,182,0.5
"""PvT""",2020-10-01 00:00:00,216,0.490741
"""PvT""",2020-11-01 00:00:00,159,0.572327
"""PvT""",2020-12-01 00:00:00,47,0.595745
"""PvT""",2021-01-01 00:00:00,106,0.5


In [6]:
sql = con.sql(f"""
    SELECT
        matchup,
        date_trunc('month', ext_datetime) AS month,
        count(*) AS games,
        avg((player_result = 'Win')::INTEGER) AS win_rate
    FROM games_1v1
    WHERE race < opponent_race AND race <> '?' AND opponent_race <> '?'
    GROUP BY ALL
    HAVING count(*) >= {MIN_GAMES}
""")
s2.same(balance, sql)

'same 201 rows, 4 columns'

In [7]:
fig = px.line(
    balance, x="month", y="win_rate", color="matchup", markers=True, hover_data=["games"],
    category_orders={"matchup": ["PvT", "PvZ", "TvZ"]},
    title="Win rate of the first race of the matchup, per month", labels={"win_rate": "win rate of the first race"},
)
fig.update_yaxes(tickformat=".0%")
fig.add_hline(y=0.5, line_color="#898781", line_width=1)
viz.show(fig, "level_2_balance_by_month")

## How long the games last, per matchup and month

The median length of the games of each matchup, including the mirrors. `games` has one row per
game with the length in minutes (its SQL twin is the `games` view).

In [8]:
length = (
    games.group_by("matchup", pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("games"), pl.col("minutes").median().alias("median_minutes"))
    .filter(pl.col("games") >= MIN_GAMES)
    .sort("matchup", "month")
    .collect()
)
length.pivot(on="matchup", index="month", values="median_minutes").sort("month")

month,PvP,PvT,PvZ,TvT,TvZ,ZvZ
datetime[ns],f64,f64,f64,f64,f64,f64
2019-08-01 00:00:00,7.68006,11.505952,9.964658,10.432292,11.028274,9.813244
2020-07-01 00:00:00,null,13.180804,11.133929,null,null,null
2020-08-01 00:00:00,8.700149,11.127604,11.524554,14.258929,13.547619,8.958705
2020-09-01 00:00:00,11.443452,11.272693,12.610119,12.816964,13.178943,8.649182
2020-10-01 00:00:00,10.525298,11.174107,11.143229,15.488095,13.248512,8.914807
2020-11-01 00:00:00,9.709821,10.630208,11.703869,11.604911,13.164435,9.15253
2020-12-01 00:00:00,9.931548,12.485863,11.853423,13.02753,13.748512,9.674107
2021-01-01 00:00:00,9.174107,10.636905,10.18378,12.765625,13.507068,9.886533
2021-02-01 00:00:00,8.306548,11.49628,10.828869,12.100446,14.793899,9.666667


In [9]:
sql = con.sql(f"""
    SELECT matchup, date_trunc('month', ext_datetime) AS month, count(*) AS games, median(minutes) AS median_minutes
    FROM games
    GROUP BY ALL
    HAVING count(*) >= {MIN_GAMES}
""")
s2.same(length, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 356 rows, 4 columns'

## The workers and the income at fixed minutes

The benchmark every coach starts with: how many workers at 6:00, 8:00 and 10:00, and the income
then. The first stats sample of each minute is the value "at" that minute (the samples are about
7 seconds apart); `sort_by().first()` in polars, `arg_min()` in SQL. The median per race and
month tells whether the economies of the games change, with the patches and the map pool.

In [10]:
MINUTES = [6, 8, 10]
at_minute = (
    stats.with_columns((s2.game_seconds() // 60).cast(pl.Int32).alias("minute"))
    .filter(pl.col("minute").is_in(MINUTES))
    .group_by("ext_fs_id", "player_id", "minute")
    .agg(
        pl.col("workers_active_count").sort_by("ext_replay_loop").first().alias("workers"),
        (pl.col("minerals_collection_rate") + pl.col("vespene_collection_rate")).sort_by("ext_replay_loop").first().alias("income"),
        (pl.col("minerals_used_current_army") + pl.col("vespene_used_current_army")).sort_by("ext_replay_loop").first().alias("army"),
    )
)
benchmarks = (
    at_minute.join(players.select("ext_fs_id", "player_id", "race", "ext_datetime"), on=["ext_fs_id", "player_id"])
    .filter(pl.col("race") != "?")
    .group_by("race", "minute", pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("players"), pl.col("workers").median().alias("median_workers"), pl.col("income").median().alias("median_income"))
    .filter(pl.col("players") >= MIN_GAMES)
    .sort("race", "minute", "month")
    .collect()
)
benchmarks.filter(pl.col("minute") == 8).pivot(on="race", index="month", values="median_workers").sort("month")

month,P,T,Z
datetime[ns],f64,f64,f64
2019-08-01 00:00:00,49.0,52.0,47.5
2020-07-01 00:00:00,49.0,42.0,47.0
2020-08-01 00:00:00,46.0,45.0,52.0
2020-09-01 00:00:00,53.0,49.5,59.0
2020-10-01 00:00:00,52.0,47.0,54.0
2020-11-01 00:00:00,52.0,46.0,53.0
2020-12-01 00:00:00,49.0,46.0,54.0
2021-01-01 00:00:00,49.0,45.0,50.0
2021-02-01 00:00:00,53.0,48.0,53.0


In [11]:
sql = con.sql(f"""
    WITH at_minute AS (
        SELECT
            ext_fs_id,
            player_id,
            floor(game_seconds(ext_replay_loop) / 60)::INTEGER AS minute,
            arg_min(workers_active_count, ext_replay_loop) AS workers,
            arg_min(minerals_collection_rate + vespene_collection_rate, ext_replay_loop) AS income,
            arg_min(minerals_used_current_army + vespene_used_current_army, ext_replay_loop) AS army
        FROM stats
        WHERE floor(game_seconds(ext_replay_loop) / 60)::INTEGER IN ({s2.sql_list(MINUTES)})
        GROUP BY ALL
    )
    SELECT
        p.race,
        a.minute,
        date_trunc('month', p.ext_datetime) AS month,
        count(*) AS players,
        median(a.workers) AS median_workers,
        median(a.income) AS median_income
    FROM at_minute a
    JOIN players p USING (ext_fs_id, player_id)
    WHERE p.race <> '?'
    GROUP BY ALL
    HAVING count(*) >= {MIN_GAMES}
""")
s2.same(benchmarks, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 663 rows, 6 columns'

In [12]:
fig = px.line(
    benchmarks, x="month", y="median_workers", color="race", facet_col="minute", markers=True, hover_data=["players"],
    color_discrete_map=viz.RACE_COLORS, category_orders={"race": ["P", "T", "Z"]},
    title="Median workers at 6:00, 8:00 and 10:00, per race and month",
)
fig.for_each_annotation(lambda annotation: annotation.update(text=f"at {annotation.text.split('=')[-1]}:00"))
viz.show(fig, "level_2_workers_by_month")

## When the upgrades finish

The median game time at which a few key upgrades finished, per month: the timing of a Stimpack or
a Warp Gate is a build order decision, and it moves with the metagame. The sprays and rewards are
upgrades too, `s2.is_cosmetic_upgrade()` removes them (the others are kept by name here anyway).

In [13]:
UPGRADES = ["Stimpack", "WarpGateResearch", "zerglingmovementspeed", "BlinkTech", "Charge", "GlialReconstitution"]
upgrade_timing = (
    upgrades.filter(pl.col("name").is_in(UPGRADES) & (pl.col("ext_replay_loop") > 0))
    .join(players.select("ext_fs_id", "player_id", "ext_datetime"), on=["ext_fs_id", "player_id"])
    .group_by(pl.col("name").alias("upgrade"), pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("games"), s2.game_seconds().median().alias("median_seconds"))
    .filter(pl.col("games") >= MIN_GAMES)
    .sort("upgrade", "month")
    .collect()
)
upgrade_timing.pivot(on="upgrade", index="month", values="median_seconds").sort("month")

month,BlinkTech,Charge,GlialReconstitution,Stimpack,WarpGateResearch,zerglingmovementspeed
datetime[ns],f64,f64,f64,f64,f64,f64
2019-08-01 00:00:00,411.830357,453.75,474.84375,449.107143,236.428571,218.258929
2020-07-01 00:00:00,447.946429,453.147321,471.272321,411.428571,239.709821,223.035714
2020-08-01 00:00:00,480.46875,483.839286,439.464286,418.080357,240.491071,222.767857
2020-09-01 00:00:00,436.428571,502.633929,446.830357,402.410714,233.459821,218.861607
2020-10-01 00:00:00,424.866071,497.678571,455.044643,426.540179,232.723214,222.321429
2020-11-01 00:00:00,450.714286,477.8125,449.821429,403.303571,234.419643,228.415179
2020-12-01 00:00:00,499.6875,477.254464,462.633929,388.973214,242.857143,231.361607
2021-01-01 00:00:00,388.571429,475.089286,463.616071,402.901786,233.861607,229.107143
2021-02-01 00:00:00,393.125,481.361607,458.482143,395.825893,236.205357,227.03125


In [14]:
sql = con.sql(f"""
    SELECT u.name AS upgrade, date_trunc('month', p.ext_datetime) AS month, count(*) AS games, median(game_seconds(u.ext_replay_loop)) AS median_seconds
    FROM upgrades u
    JOIN players p USING (ext_fs_id, player_id)
    WHERE u.name IN ({s2.sql_list(UPGRADES)}) AND u.ext_replay_loop > 0
    GROUP BY ALL
    HAVING count(*) >= {MIN_GAMES}
""")
s2.same(upgrade_timing, sql)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

'same 435 rows, 4 columns'

In [15]:
fig = px.line(
    upgrade_timing, x="month", y="median_seconds", color="upgrade", markers=True, hover_data=["games"],
    title="Median game time at which the upgrade finished, per month", labels={"median_seconds": "game seconds"},
)
viz.show(fig, "level_2_upgrade_timing")

## The map pool over the months

How many games were played on each map per month, for the most played maps: the maps enter and
leave the pool with the seasons. A heatmap of one hue, the darker the more games. In polars the long table is pivoted
with `pivot`, in DuckDB with `PIVOT ... ON month USING sum(games)`; the long form is what both
compare.

In [16]:
map_months = (
    games.group_by("title", pl.col("ext_datetime").dt.truncate("1mo").alias("month"))
    .agg(pl.len().alias("games"))
    .sort("title", "month")
    .collect()
)
MAX_MAPS = 40
map_order = (
    map_months.group_by("title").agg(pl.col("month").min().alias("first_month"), pl.col("games").sum().alias("games"))
    .top_k(MAX_MAPS, by="games")
    .sort("first_month", "games", descending=[False, True])
)
wide = map_months.pivot(on="month", index="title", values="games", sort_columns=True).fill_null(0)
wide = wide.join(map_order.select("title"), on="title", how="semi").sort(pl.col("title").replace_strict(dict(zip(map_order["title"], range(map_order.height)))))
wide.head(8)

title,2019-08-01 00:00:00.000000000,2020-07-01 00:00:00.000000000,2020-08-01 00:00:00.000000000,2020-09-01 00:00:00.000000000,2020-10-01 00:00:00.000000000,2020-11-01 00:00:00.000000000,2020-12-01 00:00:00.000000000,2021-01-01 00:00:00.000000000,2021-02-01 00:00:00.000000000,2021-03-01 00:00:00.000000000,2021-04-01 00:00:00.000000000,2021-05-01 00:00:00.000000000,2021-06-01 00:00:00.000000000,2021-07-01 00:00:00.000000000,2021-08-01 00:00:00.000000000,2021-09-01 00:00:00.000000000,2021-10-01 00:00:00.000000000,2021-11-01 00:00:00.000000000,2021-12-01 00:00:00.000000000,2022-01-01 00:00:00.000000000,2022-02-01 00:00:00.000000000,2022-03-01 00:00:00.000000000,2022-04-01 00:00:00.000000000,2022-05-01 00:00:00.000000000,2022-06-01 00:00:00.000000000,2022-07-01 00:00:00.000000000,2022-08-01 00:00:00.000000000,2022-09-01 00:00:00.000000000,2022-10-01 00:00:00.000000000,2022-11-01 00:00:00.000000000,2022-12-01 00:00:00.000000000,2023-01-01 00:00:00.000000000,2023-02-01 00:00:00.000000000,2023-03-01 00:00:00.000000000,2023-04-01 00:00:00.000000000,2023-05-01 00:00:00.000000000,2023-06-01 00:00:00.000000000,2023-07-01 00:00:00.000000000,2023-08-01 00:00:00.000000000,2023-09-01 00:00:00.000000000,2023-10-01 00:00:00.000000000,2023-11-01 00:00:00.000000000,2023-12-01 00:00:00.000000000,2024-01-01 00:00:00.000000000,2024-02-01 00:00:00.000000000,2024-03-01 00:00:00.000000000,2024-04-01 00:00:00.000000000,2024-05-01 00:00:00.000000000,2024-06-01 00:00:00.000000000,2024-07-01 00:00:00.000000000,2024-08-01 00:00:00.000000000,2024-09-01 00:00:00.000000000,2024-10-01 00:00:00.000000000,2024-11-01 00:00:00.000000000,2024-12-01 00:00:00.000000000,2025-01-01 00:00:00.000000000,2025-02-01 00:00:00.000000000,2025-03-01 00:00:00.000000000,2025-04-01 00:00:00.000000000,2025-05-01 00:00:00.000000000,2025-06-01 00:00:00.000000000,2025-07-01 00:00:00.000000000,2025-08-01 00:00:00.000000000,2025-09-01 00:00:00.000000000,2025-10-01 00:00:00.000000000,2025-11-01 00:00:00.000000000,2025-12-01 00:00:00.000000000,2026-02-01 00:00:00.000000000,2026-03-01 00:00:00.000000000,2026-04-01 00:00:00.000000000,2026-05-01 00:00:00.000000000,2026-06-01 00:00:00.000000000,2026-08-01 00:00:00.000000000
str,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
"""Deathaura LE""",0,13,103,115,176,101,56,79,88,76,9,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Pillars of Gold LE""",0,8,94,105,133,79,55,78,75,79,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Submarine LE""",0,7,46,43,64,52,26,67,42,38,10,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Eternal Empire LE""",0,11,100,169,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Ever Dream LE""",0,9,114,149,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Ice and Chrome LE""",0,4,78,117,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Romanticide LE""",0,0,0,3,140,91,66,83,78,76,63,147,64,72,111,62,103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""Oxide LE""",0,0,0,1,113,81,57,72,75,71,60,138,76,72,105,79,103,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [17]:
sql_long = con.sql("SELECT title, date_trunc('month', ext_datetime) AS month, count(*) AS games FROM games GROUP BY ALL")
print(s2.same(map_months, sql_long))
# The SQL pivot, one column per month.
con.sql("PIVOT (SELECT title, strftime(date_trunc('month', ext_datetime), '%Y-%m') AS month, ext_fs_id FROM games) ON month USING count(ext_fs_id) GROUP BY title ORDER BY title").pl().head(8)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

same 1363 rows, 3 columns


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

title,2019-08,2020-07,2020-08,2020-09,2020-10,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08,2021-09,2021-10,2021-11,2021-12,2022-01,2022-02,2022-03,2022-04,2022-05,2022-06,2022-07,2022-08,2022-09,2022-10,2022-11,2022-12,2023-01,2023-02,2023-03,2023-04,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02,2024-03,2024-04,2024-05,2024-06,2024-07,2024-08,2024-09,2024-10,2024-11,2024-12,2025-01,2025-02,2025-03,2025-04,2025-05,2025-06,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12,2026-02,2026-03,2026-04,2026-05,2026-06,2026-08
str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""10000 Feet LE""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,7,22,8,30,1,0
"""16 бит РВ""",0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""16-Bit LE""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,2,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""2.000 Atmosfere LE""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""2000 Atmospheres AIE""",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""2000 Atmospheres LE""",0,0,0,0,0,0,0,0,0,0,64,177,76,103,164,76,190,70,67,64,61,32,19,152,32,10,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""2000 Atmosphären LE""",0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,1,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""2000 Atmósferas EE""",0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [18]:
import plotly.graph_objects as go

months = [column for column in wide.columns if column != "title"]
fig = go.Figure(
    go.Heatmap(
        z=wide.select(months).to_numpy(), x=[month[:7] for month in months], y=wide["title"].to_list(),
        colorscale=[[0, "#fcfcfb"], [0.001, viz.SEQUENTIAL[0]], [1, viz.SEQUENTIAL[-1]]], xgap=2, ygap=2,
        hovertemplate="%{y}, %{x}: %{z} games<extra></extra>", colorbar=dict(title="games"),
    )
)
fig.update_layout(title="Games per map and month: the map pool", height=max(400, 22 * wide.height + 120), yaxis=dict(autorange="reversed"))
viz.show(fig, "level_2_map_pool")

## What to take from Level 2

- A measure per game (Level 1) becomes a time series when the games are grouped by their month:
  `group_by(dt.truncate("1mo"))` / `date_trunc('month')`, the median rather than the mean, and a
  minimum of games per bucket, or the series is noise.
- The entity of the series is the question: the matchup for the balance, the race for the
  economy, the map for the pool. Level 3 takes these series and asks which ones trend.
- `pivot` is for reading the table, the long form is for computing on it.

Next: [Level 3](./Level_3_Trends.ipynb).